# 🛡️ Machine Learning & NLP Project: ระบบตรวจจับข้อความมิจฉาชีพและ SMS หลอกลวง
### (AI Scam & Phishing SMS Detector using Natural Language Processing)
**หมวดวิชา:** `07_Natural_Language_Processing` + `02_Supervised_Learning`
**ชุดข้อมูล:** Thai Scam and Phishing SMS Dataset (`thai_scam_sms_dataset.csv`)

---
## 📌 วัตถุประสงค์ของโครงงาน (Objectives)
1. แก้ปัญหาภัยคุกคามทางไซเบอร์ที่กระทบต่อคนไทยทุกเพศทุกวัย โดยเฉพาะผู้สูงอายุและประชาชนทั่วไป ที่ถูกหลอกลวงผ่าน SMS และแชตแอปพลิเคชัน
2. ประยุกต์ใช้เทคนิค **Natural Language Processing (NLP)** ในการแปลงข้อความภาษาไทยด้วย **TF-IDF N-grams Vectorizer**
3. เปรียบเทียบประสิทธิภาพ 4 โมเดลการเรียนรู้ของเครื่อง:
   - **Multinomial Naive Bayes** (โมเดลมาตรฐานของการจำแนกประเภทข้อความและตัวกรองสแปม)
   - **Support Vector Machine (LinearSVC)**
   - **Logistic Regression**
   - **Random Forest Classifier**
4. สร้างโมเดลที่อธิบายเหตุผลได้ (**Explainable AI - XAI**) โดยระบุคำศัพท์และรูปแบบที่น่าสงสัยได้อย่างโปร่งใส

## 1. นำเข้าไลบรารีและโหลดข้อมูล (Import Libraries & Load Dataset)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

# โหลดชุดข้อมูลภาษาไทย
df = pd.read_csv('thai_scam_sms_dataset.csv')
print(f'จำนวนตัวอย่างทั้งหมด: {len(df)} ข้อความ')
df.head(10)

## 2. การสำรวจและประมวลผลข้อความ (Text Preprocessing & TF-IDF Extraction)
การประมวลผลภาษาธรรมชาติภาษาไทย เราใช้ `TfidfVectorizer` กำหนด `ngram_range=(1, 3)` เพื่อจับทั้งคำเดี่ยวและคำประสม เช่น 'เงินกู้', 'พัสดุตกค้าง', 'เครดิตฟรี'

In [ ]:
X = df['text']
y = df['label']

# แบ่งข้อมูล 80% Train, 20% Test (Stratified เพื่อคุมสัดส่วนคลาส)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(ngram_range=(1, 3), min_df=2, sublinear_tf=True)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f'ขนาดของ Vocabulary Matrix: {X_train_vec.shape[1]} ฟีเจอร์คำศัพท์')

## 3. ฝึกสอนและเปรียบเทียบโมเดล NLP (Model Training & Comparison)

In [ ]:
models = {
    'Multinomial Naive Bayes': MultinomialNB(alpha=0.1),
    'Logistic Regression': LogisticRegression(random_state=42, C=2.0),
    'Linear SVM': CalibratedClassifierCV(LinearSVC(random_state=42, dual=False)),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=150)
}

results = []
trained_models = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    model.fit(X_train_vec, y_train)
    trained_models[name] = model
    
    y_pred = model.predict(X_test_vec)
    y_prob = model.predict_proba(X_test_vec)[:, 1]
    
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob),
        '5-Fold CV': cross_val_score(model, X_train_vec, y_train, cv=cv).mean()
    })

df_res = pd.DataFrame(results).sort_values(by='F1-Score', ascending=False)
df_res

## 4. วิเคราะห์คำศัพท์ที่บ่งชี้ความเป็นมิจฉาชีพ (Top Indicative Scam Keywords)

In [ ]:
lr = trained_models['Logistic Regression']
feat_names = vectorizer.get_feature_names_out()
coefs = lr.coef_[0]

top_scam_indices = np.argsort(coefs)[::-1][:15]
top_scam_df = pd.DataFrame({
    'Keyword': [feat_names[i] for i in top_scam_indices],
    'Scam_Weight': [coefs[i] for i in top_scam_indices]
})

top_scam_df

## 5. ทดสอบการทำนายข้อความจริง (Inference Test)

In [ ]:
def test_sms(msg):
    nb = trained_models['Multinomial Naive Bayes']
    vec = vectorizer.transform([msg])
    pred = nb.predict(vec)[0]
    prob = nb.predict_proba(vec)[0]
    print('----------------------------------------------------')
    print(f'ข้อความ: "{msg}"')
    if pred == 1:
        print(f'🚨 ผลลัพธ์: ตรวจพบว่าเป็นมิจฉาชีพ/หลอกลวง (ความมั่นใจ {prob[1]*100:.2f}%)')
    else:
        print(f'✅ ผลลัพธ์: ข้อความปกติ ปลอดภัย (ความมั่นใจ {prob[0]*100:.2f}%)')
    print('----------------------------------------------------')

# ทดสอบข้อความมิจฉาชีพ
test_sms('Flash Express: พัสดุตกค้าง ชำระเงินเพิ่ม 20 บาทที่ bit.ly/flash-chk')
test_sms('สินเชื่อเงินด่วนทันใจ อนุมัติ 10 นาที ดอกเบี้ย 0.5% แอดไลน์ @loan99')

# ทดสอบข้อความจริง
test_sms('KBank: รหัส OTP ของคุณคือ 582914 ห้ามบอกรหัสนี้แก่ผู้อื่นทุกกรณี')
test_sms('สวัสดีครับคุณยาย พรุ่งนี้พาไปโรงพยาบาลตามหมอนัดตอน 9 โมงเช้านะครับ')